In [ ]:
# Setup and Imports

In [2]:
import requests
import pandas as pd
import time

# Your TMDB API Key
API_KEY = "bed56942f733893e88284259b5f25f48"

In [ ]:
# Fetch Genre Mapping

In [3]:
print("Fetching genre mapping...")

genre_url = f"https://api.themoviedb.org/3/genre/movie/list?api_key={API_KEY}&language=en-US"
genre_resp = requests.get(genre_url).json()

# Create a simple dictionary to map ID -> Name
genre_mapping = {}
for g in genre_resp.get('genres', []):
    genre_mapping[g['id']] = g['name']
    
print("Successfully loaded genres!")

Fetching genre mapping...
Successfully loaded genres!


In [ ]:
# Fetch Popular Movies

In [4]:
print("Starting TMDB API Extraction (2000 movies)...")
movies_data = []

# Fetch 100 pages (20 movies per page)
for page in range(1, 101):
    url = f"https://api.themoviedb.org/3/movie/popular?api_key={API_KEY}&language=en-US&page={page}"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        
        # Loop through each movie on the current page
        for item in data.get('results', []):
            
            # Find the readable genre names for this movie
            genre_names = []
            for gid in item.get('genre_ids', []):
                genre_names.append(genre_mapping.get(gid, "Unknown"))
                
            # Build a dictionary for the current movie
            movie_info = {
                "movie_id": item.get("id"),
                "title": item.get("title"),
                "release_date": item.get("release_date"),
                "genres": ", ".join(genre_names) if genre_names else "Unknown",
                "original_language": item.get("original_language"),
                "popularity_score": item.get("popularity"),
                "vote_average": item.get("vote_average"),
                "vote_count": item.get("vote_count"),
                "overview": item.get("overview")
            }
            movies_data.append(movie_info)
            
    # Show progress and pause slightly so TMDB doesn't block us
    print(f"Extracted page {page}/100...", end="\r")
    time.sleep(0.05)

Starting TMDB API Extraction (2000 movies)...
Extracted page 100/100...

In [ ]:
# Save Basic Data to CSV

In [5]:
# Convert the list of dictionaries into a Pandas DataFrame
df_raw = pd.DataFrame(movies_data)

# Save to CSV
df_raw.to_csv("tmdb_2000_movies_raw.csv", index=False)

print(f"\nDone! Extracted {len(df_raw)} records and saved to 'tmdb_2000_movies_raw.csv'.")
df_raw.head(3)


Done! Extracted 2000 records and saved to 'tmdb_2000_movies_raw.csv'.


,movie_id,title,release_date,genres,original_language,popularity_score,vote_average,vote_count,overview
0,969681,Spider-Man: Brand New Day,2026-07-29,"Science Fiction, Action, Adventure",en,839.8385,7.865,2510,Fighting crime full-time as Spider-Man in a wo...
1,1204680,Coyote vs. Acme,2026-08-20,"Comedy, Adventure, Family",en,654.7252,7.688,223,After Acme products fail him one too many time...
2,1368337,The Odyssey,2026-07-15,"Adventure, Action, Fantasy",en,464.1594,7.999,3544,"Odysseus, the legendary King of Ithaca, embark..."


In [ ]:
# Safely Add Runtimes

In [6]:
print("Loading movies to fetch runtimes. This is done safely to prevent connection errors...")

# Load the dataset we just saved
df = pd.read_csv("tmdb_2000_movies_raw.csv")
runtimes = []

for idx, row in df.iterrows():
    movie_id = row['movie_id']
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={API_KEY}"
    runtime_val = 0
    
    # Try up to 3 times if a connection hiccups
    for attempt in range(3):
        try:
            # Added a 5-second timeout to prevent getting stuck
            response = requests.get(url, timeout=5) 
            if response.status_code == 200:
                data = response.json()
                runtime_val = data.get('runtime', 0)
                break # Success! Exit the retry loop
        except Exception:
            time.sleep(1) # Wait a second before retrying
            
    runtimes.append(runtime_val)
    
    # Print progress every 100 movies
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(df)} movies...", end="\r")
        
    # Crucial: Pause slightly between requests so TMDB doesn't drop the connection
    time.sleep(0.15) 

# Add runtime column and save the final version
df['runtime'] = runtimes
df.to_csv("tmdb_2000_movies_raw.csv", index=False)

print("\n\nSuccess! Runtime successfully added to your dataset.")
df[['title', 'runtime']].head(5)

Loading movies to fetch runtimes. This is done safely to prevent connection errors...
Processed 2000/2000 movies...

Success! Runtime successfully added to your dataset.


,title,runtime
0,Spider-Man: Brand New Day,145
1,Coyote vs. Acme,103
2,The Odyssey,173
3,Mutiny,95
4,Colony,123
